# 第 26 章　実験と因果推論

図書館で追加の通知を送った人の方が，期限までに返却する割合が低く見えました。通知を送る人をどう決めていたのでしょうか。講義では春の蚊の実験に戻り，条件の分け方が平均の差にどんな意味を持たせるかを調べます。後半では別の架空の図書館の表を読み，通知の効果を確かめる比較を提案します。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。ローカルでは配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置きます。ファイルが見つからなければ公開 URL から読み込みます。

## 講義　何の差を知りたいか

### 春の実験を読む

成人男性 43 人を，現地のビールを飲む 25 人と水を飲む 18 人に無作為に分けた実験です。量はどちらも 1 L で，飲用前と飲用 15 分後に測定を始めました。二つのトラップに入った合計数が `activated` です。まず `test` が `after` の一人一行を使い，人数と平均を求めます。同じ時点の二群を比べるための数値です。

In [ ]:
from pathlib import Path

import pandas as pd

file_name = "mosquito-experiment.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

experiment = pd.read_csv(csv_path)
after = experiment.loc[experiment["test"] == "after"]
post_summary = after.groupby("group")["activated"].agg(
    people="count",
    total="sum",
    mean_caught="mean",
)
post_summary.round(2)

### 同じ人の二条件

`subj1` は飲用前に 16 匹，ビールを飲んだ後の測定では 14 匹でした。知りたい「ビールの後 − 水の後」の二列を，同じ測定時点について書きます。

| 人 | 水を飲んだ場合の 15 分後 | ビールを飲んだ場合の 15 分後 |
| --- | ---: | ---: |
| subj1 | 未観測 | 14 匹 |

各条件に対応する結果を**潜在結果**と呼びます。飲用前の 16 匹を左に置くと，どの時点の差を求めたことになりますか。平均効果は，同じ参加者全員について二条件の差を一人ずつ求め，平均した値です。実際の表では各人に割り付けられた条件の結果が分かります。

### 架空の四人をくじで分ける

架空の例として，水の場合とビールの場合の値を両方作りました。右端は一人ずつ引いた差で，四人の平均は +4 匹です。A・B にビール，C・D に水を割り当てるとしたら，どのセルを観測し，群平均差はどうなりますか。実行する前に求めてください。

| 人 | 水なら（匹） | ビールなら（匹） | ビール − 水 |
| --- | ---: | ---: | ---: |
| A | 10 | 14 | +4 |
| B | 14 | 20 | +6 |
| C | 18 | 20 | +2 |
| D | 22 | 26 | +4 |

In [ ]:
potential = pd.DataFrame(
    {"water": [10, 14, 18, 22], "beer": [14, 20, 20, 26]},
    index=["A", "B", "C", "D"],
)

beer_group = potential.index.isin(["A", "B"])
beer_observed = potential.loc[beer_group, "beer"]
water_observed = potential.loc[~beer_group, "water"]
print(beer_observed.to_dict(), water_observed.to_dict())
print(f"Beer minus water: {beer_observed.mean() - water_observed.mean():+.1f}")

次は，ビールへ割り付ける二人を変えます。例えば `"AC"` を `list("AC")` にすると，一人ずつ A・C のラベルを調べられます。`isin()` が真になる人はビール列を，残りの人は水列を読みます。ビールに入る組を替えると，値を取り出す列も替わることを確認します。

In [ ]:
assignments = ["AB", "AC", "AD", "BC", "BD", "CD"]
gaps = []
for pair in assignments:
    beer_group = potential.index.isin(list(pair))
    mean_beer = potential.loc[beer_group, "beer"].mean()
    mean_water = potential.loc[~beer_group, "water"].mean()
    gaps.append(mean_beer - mean_water)

assignment_gaps = pd.Series(gaps, index=assignments, name="observed_gap")
print(f"Mean across six assignments: {assignment_gaps.mean():+.1f}")
assignment_gaps

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 3.8), layout="constrained")
ax.axvline(0, color="#687581", linestyle=":", linewidth=1.2, label="Zero difference")
ax.axvline(assignment_gaps.mean(), color="#a7421f", linestyle="--", linewidth=1.5,
           label="Average effect (+4)")
ax.scatter(assignment_gaps, assignment_gaps.index, color="#195b8a", s=50, zorder=3)
ax.invert_yaxis()
ax.set(xlim=(-5, 13), xticks=[-4, 0, 4, 8, 12],
       xlabel="Difference in group means (mosquitoes)", ylabel="Assigned to beer")
ax.grid(axis="x", color="#e1e7eb", zorder=0)
ax.legend(loc="upper right", frameon=False)
plt.show()

六通りが等確率なら，群平均差の六つの値を同じ重さで平均できます。A がビールになる組と水になる組は何通りずつですか。四人全員の両条件が平均にどう入るかを説明してください。

春に実験の捕獲数を固定して札だけ動かせたのは，一人ずつ二条件が同じと仮定していたためです。今回の架空例では，A の行を見ても，条件を替えると読む値が変わります。

### 実験で観測した前後と二群

一人の前後を `volunteer` で結合してから，後の捕獲数から前を引きます。前後，二群，前後差を二群で比較することは，それぞれどの引き算に対応するでしょうか。図で線の向きと点の高さを照らし合わせます。

In [ ]:
before = experiment.loc[experiment["test"] == "before", ["volunteer", "activated"]]
paired = after[["volunteer", "group", "activated"]].merge(
    before, on="volunteer", suffixes=("_after", "_before"), validate="one_to_one",
)
paired["change"] = paired["activated_after"] - paired["activated_before"]
paired.head(3)

In [ ]:
group_means = paired.groupby("group").agg(
    people=("volunteer", "count"),
    before=("activated_before", "mean"),
    after=("activated_after", "mean"),
    change=("change", "mean"),
)
experiment_gaps = group_means.loc["beer", ["before", "after", "change"]] - group_means.loc[
    "water", ["before", "after", "change"]
]
group_means.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.9), layout="constrained")
for group, color, marker, style in [
    ("beer", "#a7421f", "o", "-"),
    ("water", "#195b8a", "s", "--"),
]:
    row = group_means.loc[group]
    means = row[["before", "after"]]
    ax.plot([0, 1], means, color=color, marker=marker, linestyle=style,
            label=f"{group.title()} (n = {int(row['people'])})", markersize=7)
    ax.annotate(f"{means['before']:.2f}", (0, means["before"]), xytext=(-11, 0),
                textcoords="offset points", ha="right", va="center", color=color)
    ax.annotate(f"{means['after']:.2f}", (1, means["after"]), xytext=(11, 0),
                textcoords="offset points", ha="left", va="center", color=color)
ax.set(xlim=(-0.23, 1.23), ylim=(14, 26),
       ylabel="Mean mosquitoes caught per participant")
ax.set_xticks([0, 1], ["Before drinking", "15 minutes after"])
ax.grid(axis="y", color="#e1e7eb")
ax.legend(loc="upper left", frameon=False)
plt.show()

In [ ]:
experiment_gaps.rename("beer_minus_water").round(2)

+6.24 匹はビール群の前後差，+4.43 匹は飲用後の二群の差，+5.80 匹は二群の前後差を比べた値です。+4.43 匹を平均効果の推定に使う根拠は，飲み物を無作為に割り付けたという原著の記載です。一回の群平均差は，どの人がどちらの条件に入ったかによって変わります。春には全員の効果が 0 と仮定し，今回ほどの差がどのくらい出るかを検定しました。

### 通知する相手が違うとき

ここからは架空の図書館です。追加通知を送る相手は，過去に返却が遅れた人を優先していました。通知の有無ごとに人数と期限内返却数を足し，まず全体の割合を求めます。

In [ ]:
library = pd.DataFrame({
    "late_before": ["yes", "yes", "no", "no"],
    "notice": ["sent", "not sent", "sent", "not sent"],
    "users": [80, 20, 20, 80],
    "on_time": [48, 10, 18, 64],
})

overall = library.groupby("notice")[["users", "on_time"]].sum()
overall["rate"] = overall["on_time"] / overall["users"]
overall

全体の差を読んだら，通知あり・なしの 100 人ずつに，過去に延滞した人が何人含まれるかを比べます。次に延滞歴をそろえて，どの人数を分母にするかを確認します。下の図の左は人数，右は各延滞歴での割合です。

In [ ]:
library["rate"] = library["on_time"] / library["users"]
library

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.3, 4.25), layout="constrained")
notice_order = ["not sent", "sent"]
late_users = library.loc[library["late_before"] == "yes"].set_index("notice")["users"]
late_users = late_users.loc[notice_order]
no_late_users = overall.loc[notice_order, "users"] - late_users
axes[0].bar([0, 1], late_users, width=0.56, color="#195b8a", label="Prior late return")
axes[0].bar([0, 1], no_late_users, bottom=late_users, width=0.56,
            color="#a4c6da", edgecolor="#4f7f98", hatch="//", label="No prior late return")
for position, lower in enumerate(late_users):
    axes[0].text(position, lower / 2, f"{lower:d}", ha="center", va="center", color="white")
    upper = no_late_users.iloc[position]
    axes[0].text(position, lower + upper / 2, f"{upper:d}", ha="center", va="center",
                 color="#182129")
axes[0].set(ylabel="Users", ylim=(0, 110))
axes[0].set_xticks([0, 1], ["No notice", "Notice"])
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), frameon=False)

for status, offset, color, marker, label in [
    ("not sent", -0.12, "#195b8a", "s", "No notice"),
    ("sent", 0.12, "#a7421f", "o", "Notice"),
]:
    rates = library.loc[library["notice"] == status].set_index("late_before")["rate"]
    rates = rates.loc[["yes", "no"]] * 100
    x = [offset, 1 + offset]
    axes[1].scatter(x, rates, color=color, marker=marker, s=49, label=label, zorder=3)
    for tick, rate in zip(x, rates):
        axes[1].annotate(f"{rate:.0f}%", (tick, rate), xytext=(0, 7),
                         textcoords="offset points", ha="center", color=color)
axes[1].set(ylabel="Returned on time (%)", xlim=(-0.42, 1.42), ylim=(0, 104))
axes[1].set_xticks([0, 1], ["Prior late return", "No prior late return"])
axes[1].grid(axis="y", color="#e1e7eb", zorder=0)
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncols=2, frameon=False)
plt.show()

通知あり・なしの双方に，延滞歴のある人とない人が半分ずついる構成を考えます。`pivot()` で行に延滞歴，列に通知を置き，二行を半分ずつ足します。通知の有無による二つの割合を，どちらも同じ延滞歴の構成にそろえた観察上の比較です。

In [ ]:
rates_by_history = library.pivot(index="late_before", columns="notice", values="rate")
same_mix = 0.5 * rates_by_history.loc["yes"] + 0.5 * rates_by_history.loc["no"]
same_mix.rename("rate_with_same_mix")

延滞歴をそろえても，担当者が普段は返却が難しそうな人をどう選んだかなど，ほかの事情が残り得ます。その事情と通知の効果を区別したいときは，同じ時期に，利用者単位で通常の案内のみの条件と，通常の案内に追加通知を一度加える条件へ無作為に分ける案が考えられます。実習では別の表について，その比較で何を知りたいかを決めます。

## 実習　別の通知表から比較を作る（約 20 分）

### 1. 予想と問いを書く（4 分）

別の架空の図書館の 180 人です。一人について指定した貸出一件を調べます。まず下の表で，過去の延滞歴別に，誰が追加通知を受けたかを読みます。まだ今回の返却割合は表示しません。

In [ ]:
practice_groups = pd.DataFrame({
    "late_before": ["yes", "yes", "no", "no"],
    "notice": ["sent", "not sent", "sent", "not sent"],
    "users": [30, 60, 80, 10],
})
practice_groups

全体の通知あり・なしの差は，延滞歴をそろえた差より大きく見えると思いますか。どの人に通知が多かったかを根拠に，予想してください。次に知りたい対象を，この図書館で通知を受け取れる利用者全体，または延滞歴のある利用者から一つ選び，追加通知の効果を一文で問います。

**予想とその理由**　…

**選んだ対象と問い**　…

### 2. 期限内返却の割合を出す（6 分）

`practice_counts` の `on_time` は期限内に返した人数，`users` はその組の人数です。次のセルの `practice_rates` に入っている `None` を，割合を求める式に替えます。下に出る二つの表で，分母と割合を読んでください。通知ありとなしを引くと，全体と，延滞歴をそろえた場合とで差はどう違いますか。

In [ ]:
practice_counts = practice_groups.assign(on_time=[15, 24, 56, 6])
practice_rates = None  # None を，期限内返却の割合を求める式に置き換えます。

if practice_rates is None:
    print("practice_rates に割合を求める式を書いてください。")
else:
    practice_by_history = practice_counts.assign(rate=practice_rates)
    practice_overall = practice_counts.groupby("notice")[["users", "on_time"]].sum()
    practice_overall["rate"] = practice_overall["on_time"] / practice_overall["users"]
    display(practice_overall.round(3))
    display(practice_by_history.round(3))

**分母と差を読んだ結果**　…

### 3. 比較を提案する（7 分）

ここでは比較の計画を作り，利用者への通知は送りません。実際に人を対象とする調査・実験を行う場合は，事前に担当教員と必要な学内手続きを確認し，参加の任意性，説明と同意，個人情報，通常のサービスへの影響を検討します。実施が認められていない計画は，計画の検討または既存データの分析にとどめます。

最初に選んだ対象に対して，通知を送ったときの効果を調べる比較を作ります。これまでの結果を説明できる可能性として，「通知が役立つこと」と「通知を受ける人の事情が違うこと」を考え，この比較で確かめられることを書きます。

| 決めること | 自分の比較案 |
| --- | --- |
| 対象は誰か | … |
| どの二条件にするか | … |
| 何を単位に，いつ，どう分けるか | … |
| 何を，いつの結果で判断するか | … |
| この比較で，どの説明を確かめられるか | … |

**この比較で，二条件にどんな結果を予想するか**　…

### 4. 結果を見て考えを変える（3 分）

比較を実施できたとしたら，どんな結果が見えれば，手順 3 で書いた予想を見直しますか。比べる数字を一つ示し，差の大きさや不確かさについて次に調べたいことを一つ書きます。

**考えを変える結果と，確かめたいこと**　…

最後にカーネルを再起動して，全セルを上から実行します。最初の予想，選んだ問い，割合の表，比較案と短い説明が残る状態でノートを保存します。

### データの出典

実験データは第一著者 Thierry Lefèvre が提供し，Matthew Brett が配布した CC BY データです。図書館の人数は教材用に作った架空の値です。 [原著](https://doi.org/10.1371/journal.pone.0009546) ・ [実験データの来歴と利用条件](https://github.com/matthew-brett/datasets/tree/master/mosquito_beer)